In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_4_classes', 'seed': 42, 'n_classes': 4}, 'dataset': {'split_type': 'test'}, 'paths': {'data_exploration_dir': 'output/experiment_with_4_classes/data_exploration', 'embeddings_dir': 'output/experiment_with_4_classes/embeddings', 'models_dir': 'output/experiment_with_4_classes/models', 'predictions_dir': 'output/experiment_with_4_classes/predictions', 'results_dir': 'output/experiment_with_4_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_SPLIT_TYPE: test

[EVALUATION]
  EVALUATIO

# Reuters News Topic Classification - Exploratory Data Analysis

This notebook performs detailed exploratory data analysis on the Reuters news topic classification dataset. We'll analyze various aspects of the data to better understand its characteristics and potential challenges.

## Setup and Data Loading

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from src.datasets.dataset import get_dataset
from src.exploration import class_frequency, vocabulary_drift, length_distribution


In [3]:

# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 4
INFO |   - Samples per class: 20
INFO |   - Random seed: None
INFO | Loading small test dataset with 20 samples per class across 4 classes
INFO | Selected classes: earn, acq, crude, interest
INFO |   - Class 'earn': 14 train, 6 test
INFO |   - Class 'acq': 14 train, 6 test
INFO |   - Class 'crude': 14 train, 6 test
INFO |   - Class 'interest': 14 train, 6 test


Loaded 56 training documents with 4 classes


## 1. Class Distribution Analysis

Let's analyze the distribution of topics in our dataset to understand class imbalance.

In [4]:

# Analyze class distribution
print("\n=== Class Distribution Analysis ===")
class_stats = class_frequency(
    labels=y_train,
    plot=True,
    save_path=os.path.join(DATA_EXPLORATION_DIR, "class_distribution.png"),
    top_n=N_CLASSES
)


=== Class Distribution Analysis ===


## 2. Document Length Analysis

Understanding the length distribution of articles helps us make decisions about preprocessing and model architecture.

In [5]:
# Plot length distribution
plt.figure(figsize=(10, 6))


stats = length_distribution(X_train, save_path=os.path.join(DATA_EXPLORATION_DIR, "document_length_distribution.png"), output_dir=DATA_EXPLORATION_DIR)
print(f"Mean length: {stats['stats']['mean']:.1f} tokens, median: {stats['stats']['median']}")

# Access percentile information from the returned stats
percentile_stats = stats['percentile_stats']


Document Length Percentiles:
25th percentile: 56 tokens
50th percentile: 98 tokens
75th percentile: 233 tokens
90th percentile: 429 tokens
95th percentile: 487 tokens
99th percentile: 567 tokens
Mean length: 165.7 tokens, median: 98.5


<Figure size 1000x600 with 0 Axes>

In [6]:
import os

# Plot length distribution
plt.figure(figsize=(10, 6))
stats = length_distribution(X_train, save_path=os.path.join(DATA_EXPLORATION_DIR, "document_length_distribution.png"))
print(f"Mean length: {stats['stats']['mean']:.1f} tokens, median: {stats['stats']['median']}")

# Calculate length percentiles
lengths = [len(doc.split()) for doc in X_train]
percentiles = np.percentile(lengths, [25, 50, 75, 90, 95, 99])
print("\nDocument Length Percentiles:")
length_stats = []
for p, percentile in zip([25, 50, 75, 90, 95, 99], percentiles):
    print(f"{p}th percentile: {percentile:.0f} tokens")
    length_stats.append({'percentile': p, 'length': int(percentile)})

# Save document length statistics to CSV
pd.DataFrame(length_stats).to_csv(os.path.join(DATA_EXPLORATION_DIR, "document_length_stats.csv"), index=False)

Mean length: 165.7 tokens, median: 98.5

Document Length Percentiles:
25th percentile: 57 tokens
50th percentile: 98 tokens
75th percentile: 234 tokens
90th percentile: 430 tokens
95th percentile: 488 tokens
99th percentile: 567 tokens


<Figure size 1000x600 with 0 Axes>

## 3. Vocabulary Analysis

Let's examine the vocabulary characteristics of our dataset.

In [7]:
#!/usr/bin/env python
"""
Simple script to run comprehensive vocabulary analysis on Reuters data.
"""
from src.exploration import comprehensive_analysis


# Run comprehensive analysis with custom settings
analysis_results = comprehensive_analysis(
    texts=X_train,                                          # Your text documents
    labels=y_train,                                         # Class labels for class-specific analysis
    label_names=classes,                                # Names of the classes 
    output_dir=os.path.join(DATA_EXPLORATION_DIR),                    # Output directory for results
    min_word_length=3,                                      # Minimum word length (filters short tokens)
    top_n=50,                                               # Number of top words to analyze
    create_visualizations=True,                             # Create and save plots
    create_csv=True                                         # Save results to CSV files
)

# The analysis_results dictionary contains all the results:
# - analysis_results['basic'] - basic analysis (only stopwords removed)
# - analysis_results['standard'] - standard filtering (stopwords, numbers, financial terms)
# - analysis_results['advanced'] - advanced filtering (adds domain-specific stopwords)
# - analysis_results['basic_class'] - class-specific words with basic filtering
# - analysis_results['standard_class'] - class-specific words with standard filtering
# - analysis_results['advanced_class'] - class-specific words with advanced filtering

print("\nAnalysis complete! Results saved to the 'vocab_analysis_custom' directory.")
print("You can also access the results programmatically through the returned dictionary.")

# Example: Get the top 5 words after advanced filtering
print("\nTop 5 words (advanced filtering):")
for word, count in analysis_results['advanced']['word_counts'].most_common(5):
    print(f"  {word}: {count:,}")

# Example: Get the most distinctive word for each class
print("\nMost distinctive word by class (advanced filtering):")
for class_name in classes:
    if class_name in analysis_results['advanced_class'].columns:
        top_word = analysis_results['advanced_class'][class_name][0]
        if top_word:  # Check that it's not an empty string
            print(f"  {class_name}: {top_word}") 

Running comprehensive vocabulary analysis on 56 documents...

======= BASIC FILTERING =======
Total unique words (basic filtering): 2,187

======= STANDARD FILTERING =======
Total unique words (standard filtering): 1,944

======= ADVANCED FILTERING =======
Total unique words (advanced filtering): 1,944
Visualization saved to 'top_words_comprehensive.png'

======= CLASS-SPECIFIC ANALYSIS =======

Analysis complete! Results saved to the 'vocab_analysis_custom' directory.
You can also access the results programmatically through the returned dictionary.

Top 5 words (advanced filtering):
  bank: 52
  oil: 43
  year: 40
  rate: 39
  would: 32

Most distinctive word by class (advanced filtering):
  earn: net (28)
  acq: share (19)
  crude: oil (43)
  interest: bank (37)


## 4. Vocabulary drift



In [8]:

# Analyze vocabulary drift between train and test sets
print("\n=== Vocabulary Drift Analysis ===")
vocab_drift = vocabulary_drift(
    train_texts=X_train,
    test_texts=X_test,
    top_k=2000,
    min_freq=100,
    output_path=Path(RESULTS_DIR) / "vocabulary_drift.csv"
)

# Display top 10 tokens with highest drift
print("\nTop 20 tokens with highest frequency drift:")
display(vocab_drift.head(20))

# Display class distribution statistics
print("\nClass distribution statistics:")
display(class_stats['counts'])


=== Vocabulary Drift Analysis ===

Top 20 tokens with highest frequency drift:


,token,train_freq,test_freq,abs_diff
1,mln,0.785340,0.214660,0.570681
6,of,0.765766,0.234234,0.531532
0,and,0.757576,0.242424,0.515152
8,the,0.750397,0.249603,0.500795
7,it,0.745098,0.254902,0.490196
2,a,0.745020,0.254980,0.490040
9,for,0.741007,0.258993,0.482014
5,in,0.735043,0.264957,0.470085
4,to,0.717868,0.282132,0.435737
3,said,0.685535,0.314465,0.371069



Class distribution statistics:


earn        14
acq         14
crude       14
interest    14
Name: count, dtype: int64